# Step 01: Exploratory Data Analysis (EDA) of Annotations
### Statistical Profiling of Discovered Categories, Dimensions, Aspect Ratios & Tag Densities
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 4, 5, 6, 13).

### Key Analyses:
- Discovers annotation JSON files from `coco_files/`.
- Parses and prints category distribution with percentages and bar chart.
- Computes bounding box dimensions (width, height, area, aspect ratio).
- Analyzes tag density per image (single-tag vs multi-tag shelf scenes).
- Displays sample annotations per class in-memory (using Pure PIL & IPython - zero matplotlib disk leaks).

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
print_vram_usage("Init")
logger.info("=" * 65)

import io
def show_plt_figure(fig=None):
    """Guarantees inline plot display in Jupyter under headless and Agg backends."""
    if fig is None:
        fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    pil_img = Image.open(buf)
    plt.close(fig)
    display(pil_img)


In [ ]:
# CELL 2 - Path & Annotation Fields Configuration
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

# Auto-discover active images pool directory across standard pipeline locations
candidate_image_dirs = [
    REPO_ROOT / "multi_class_train_rfdetr" / "images",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "images",
    Path("images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])
os.makedirs(IMAGES_DIR, exist_ok=True)

# COCO annotation field mappings
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

logger.info(f"Input JSON Directory: {INPUT_JSON_DIR.resolve()}")
logger.info(f"Active Images Pool:   {IMAGES_DIR.resolve()} ({len(list(IMAGES_DIR.glob('*'))):,} cached images)")


In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in tqdm(json_files, desc="[CELL 4] Reading JSON files", unit="file"):
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in tqdm(raw_records, desc="[CELL 5] Parsing annotations", unit="rec"):
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")
# Construct definitions for downstream EDA visualizers and exports
categories_list = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]
annotations_by_image = image_annotations
annotations_by_cat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
for anns in image_annotations.values():
    for a in anns:
        annotations_by_cat[a["category_name"]].append(a)

In [ ]:
# CELL 5 - Category Distribution Bar & Share Donut Chart
if total_boxes > 0:
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    cat_names = [c["name"] for c in categories_list]
    cat_counts = [len(annotations_by_cat[c["name"]]) for c in categories_list]
    palette = ["#1976D2", "#388E3C", "#F57C00", "#7B1FA2", "#D32F2F", "#00796B"][:len(cat_names)]
    
    # 1. Bar Chart with Percentages
    bars = axs[0].bar(cat_names, cat_counts, color=palette, edgecolor="black", alpha=0.85)
    axs[0].set_title("Annotation Count per Discovered Category", fontsize=13, fontweight="bold")
    axs[0].set_xlabel("Category", fontsize=11)
    axs[0].set_ylabel("Bounding Box Count", fontsize=11)
    axs[0].grid(axis="y", linestyle="--", alpha=0.5)
    for bar in bars:
        yval = bar.get_height()
        pct = (yval / total_boxes) * 100
        axs[0].text(bar.get_x() + bar.get_width()/2.0, yval + max(1, total_boxes * 0.01), f"{yval} ({pct:.1f}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")
    
    # 2. Donut Chart of Class Proportions
    wedges, texts, autotexts = axs[1].pie(
        cat_counts,
        labels=cat_names,
        autopct="%1.1f%%",
        colors=palette,
        startangle=140,
        pctdistance=0.8,
        wedgeprops=dict(width=0.4, edgecolor="white", linewidth=2)
    )
    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight("bold")
    axs[1].set_title("Dataset Class Share Composition", fontsize=13, fontweight="bold")
    
    plt.tight_layout()
    show_plt_figure()


In [ ]:
# CELL 6 - Bounding Box Scale, Aspect Ratio & Area Distributions
if total_boxes > 0:
    widths, heights, aspect_ratios, areas = [], [], [], []
    for cat_name, anns in annotations_by_cat.items():
        for a in anns:
            bbox = a.get("bbox", [])
            if len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
                w, h = float(bbox[2]), float(bbox[3])
                widths.append(w)
                heights.append(h)
                aspect_ratios.append(w / max(1.0, h))
                areas.append(w * h)
                
    widths_arr = np.array(widths)
    heights_arr = np.array(heights)
    aspect_ratios_arr = np.array(aspect_ratios)
    areas_arr = np.array(areas)

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    
    # Panel 1: Width
    axs[0, 0].hist(widths_arr, bins=35, color="#1976D2", edgecolor="black", alpha=0.8)
    axs[0, 0].axvline(np.median(widths_arr), color="red", linestyle="--", linewidth=2, label=f"Median: {np.median(widths_arr):.1f} px")
    axs[0, 0].set_title("Bounding Box Width Distribution", fontsize=12, fontweight="bold")
    axs[0, 0].set_xlabel("Width (px)")
    axs[0, 0].set_ylabel("Count")
    axs[0, 0].grid(True, linestyle=":", alpha=0.5)
    axs[0, 0].legend()
    
    # Panel 2: Height
    axs[0, 1].hist(heights_arr, bins=35, color="#388E3C", edgecolor="black", alpha=0.8)
    axs[0, 1].axvline(np.median(heights_arr), color="red", linestyle="--", linewidth=2, label=f"Median: {np.median(heights_arr):.1f} px")
    axs[0, 1].set_title("Bounding Box Height Distribution", fontsize=12, fontweight="bold")
    axs[0, 1].set_xlabel("Height (px)")
    axs[0, 1].set_ylabel("Count")
    axs[0, 1].grid(True, linestyle=":", alpha=0.5)
    axs[0, 1].legend()
    
    # Panel 3: Aspect Ratio (w / h)
    axs[1, 0].hist(aspect_ratios_arr, bins=35, color="#F57C00", edgecolor="black", alpha=0.8)
    axs[1, 0].axvline(np.median(aspect_ratios_arr), color="red", linestyle="--", linewidth=2, label=f"Median: {np.median(aspect_ratios_arr):.2f}")
    axs[1, 0].set_title("Aspect Ratio Distribution (Width / Height)", fontsize=12, fontweight="bold")
    axs[1, 0].set_xlabel("Aspect Ratio")
    axs[1, 0].set_ylabel("Count")
    axs[1, 0].grid(True, linestyle=":", alpha=0.5)
    axs[1, 0].legend()
    
    # Panel 4: Area (px^2)
    axs[1, 1].hist(areas_arr, bins=35, color="#7B1FA2", edgecolor="black", alpha=0.8)
    axs[1, 1].axvline(np.median(areas_arr), color="red", linestyle="--", linewidth=2, label=f"Median: {np.median(areas_arr):.0f} px²")
    axs[1, 1].set_title("Bounding Box Area Distribution", fontsize=12, fontweight="bold")
    axs[1, 1].set_xlabel("Area (px²)")
    axs[1, 1].set_ylabel("Count")
    axs[1, 1].grid(True, linestyle=":", alpha=0.5)
    axs[1, 1].legend()
    
    plt.tight_layout()
    show_plt_figure()
    
    logger.info(f"Median Box Width:       {np.median(widths_arr):.1f} px (Mean: {np.mean(widths_arr):.1f} px)")
    logger.info(f"Median Box Height:      {np.median(heights_arr):.1f} px (Mean: {np.mean(heights_arr):.1f} px)")
    logger.info(f"Median Aspect Ratio:    {np.median(aspect_ratios_arr):.2f} (Mean: {np.mean(aspect_ratios_arr):.2f})")
    logger.info(f"Median Box Area:        {np.median(areas_arr):.0f} px² (Mean: {np.mean(areas_arr):.0f} px²)")


In [ ]:
# CELL 7 - Spatial Center Heatmap & Dimensions Scatter by Class
if total_boxes > 0:
    fig, axs = plt.subplots(1, 2, figsize=(16, 6))
    
    center_xs, center_ys = [], []
    box_w_by_cat = defaultdict(list)
    box_h_by_cat = defaultdict(list)
    
    for cname, anns in annotations_by_cat.items():
        for a in anns:
            bbox = a.get("bbox", [])
            if len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
                cx = bbox[0] + bbox[2] / 2.0
                cy = bbox[1] + bbox[3] / 2.0
                center_xs.append(cx)
                center_ys.append(cy)
                box_w_by_cat[cname].append(bbox[2])
                box_h_by_cat[cname].append(bbox[3])
                
    # 1. 2D Spatial Center Heatmap
    if center_xs and center_ys:
        h2d, xedges, yedges, img = axs[0].hist2d(center_xs, center_ys, bins=35, cmap="YlOrRd")
        axs[0].set_title("Spatial Center Density Heatmap (cx, cy)", fontsize=13, fontweight="bold")
        axs[0].set_xlabel("X Center Coordinate (px)")
        axs[0].set_ylabel("Y Center Coordinate (px)")
        axs[0].invert_yaxis()  # Image coordinate system: origin at top-left
        cbar = plt.colorbar(img, ax=axs[0])
        cbar.set_label("Box Count Density", fontsize=10)
    
    # 2. Width vs Height Scatter colored by Category
    class_colors = {"location_tag": "#1976D2", "blue_aisle": "#388E3C", "blue_bay": "#F57C00"}
    default_palette = ["#7B1FA2", "#D32F2F", "#00796B"]
    for idx, (cname, ws) in enumerate(box_w_by_cat.items()):
        hs = box_h_by_cat[cname]
        color = class_colors.get(cname, default_palette[idx % len(default_palette)])
        axs[1].scatter(ws, hs, label=f"{cname} ({len(ws)})", color=color, alpha=0.55, edgecolors="none", s=28)
        
    axs[1].set_title("Bounding Box Width vs Height by Category", fontsize=13, fontweight="bold")
    axs[1].set_xlabel("Width (px)")
    axs[1].set_ylabel("Height (px)")
    axs[1].grid(True, linestyle="--", alpha=0.5)
    axs[1].legend(loc="upper right", frameon=True)
    
    plt.tight_layout()
    show_plt_figure()


In [ ]:
# CELL 8 - Category Boxplots & Multi-Tag Co-Occurrence Heatmap
if total_boxes > 0:
    fig, axs = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. Per-Class Boxplot (Aspect Ratio Distribution)
    cat_names_sorted = [c["name"] for c in categories_list if c["name"] in annotations_by_cat]
    ar_data = []
    for cname in cat_names_sorted:
        ars = [a["bbox"][2] / max(1.0, a["bbox"][3]) for a in annotations_by_cat[cname] if len(a.get("bbox", [])) == 4 and a["bbox"][3] > 0]
        ar_data.append(ars)
        
    if ar_data:
        bp = axs[0].boxplot(ar_data, tick_labels=cat_names_sorted, patch_artist=True, showfliers=False)
        colors = ["#42A5F5", "#66BB6A", "#FFA726", "#AB47BC", "#EF5350"]
        for patch, col in zip(bp["boxes"], colors * 3):
            patch.set_facecolor(col)
            patch.set_alpha(0.7)
        axs[0].set_title("Aspect Ratio (Width / Height) by Category", fontsize=13, fontweight="bold")
        axs[0].set_ylabel("Aspect Ratio")
        axs[0].grid(True, linestyle=":", alpha=0.6)
        axs[0].tick_params(axis="x", rotation=15)
        
    # 2. Multi-Class Co-occurrence Matrix
    N = len(cat_names_sorted)
    co_matrix = np.zeros((N, N), dtype=int)
    cat_to_idx = {name: i for i, name in enumerate(cat_names_sorted)}
    
    for anns in annotations_by_image.values():
        present_cats = set(a["category_name"] for a in anns if "category_name" in a and a["category_name"] in cat_to_idx)
        for c1 in present_cats:
            for c2 in present_cats:
                co_matrix[cat_to_idx[c1], cat_to_idx[c2]] += 1
                
    im = axs[1].imshow(co_matrix, cmap="Blues")
    axs[1].set_xticks(range(N))
    axs[1].set_yticks(range(N))
    axs[1].set_xticklabels(cat_names_sorted, rotation=20, ha="right")
    axs[1].set_yticklabels(cat_names_sorted)
    axs[1].set_title("Category Co-Occurrence Across Images", fontsize=13, fontweight="bold")
    
    for i in range(N):
        for j in range(N):
            val = co_matrix[i, j]
            color = "white" if val > (co_matrix.max() * 0.6) else "black"
            axs[1].text(j, i, str(val), ha="center", va="center", color=color, fontweight="bold")
            
    cbar = plt.colorbar(im, ax=axs[1])
    cbar.set_label("Shared Images Count", fontsize=10)
    
    plt.tight_layout()
    show_plt_figure()


In [ ]:
# CELL 9 - Tag Density per Image Distribution & Histogram
if annotations_by_image:
    tags_per_img = [len(anns) for anns in annotations_by_image.values()]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    max_tags = max(tags_per_img) if tags_per_img else 1
    bins = range(1, max_tags + 2)
    ax.hist(tags_per_img, bins=bins, align="left", color="#3F51B5", edgecolor="black", alpha=0.85, rwidth=0.8)
    ax.set_title("Tags / Detections Density per Image Distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel("Number of Annotated Tags per Image", fontsize=11)
    ax.set_ylabel("Image Count", fontsize=11)
    ax.set_xticks(range(1, min(max_tags + 1, 25)))
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    show_plt_figure()

    logger.info("=" * 70)
    logger.info(" TAG DENSITY PROFILE PER IMAGE")
    logger.info("=" * 70)
    logger.info(f"   - Total unique images:           {len(tags_per_img)}")
    logger.info(f"   - Single-tag images (count=1):   {sum(1 for c in tags_per_img if c == 1)} ({sum(1 for c in tags_per_img if c == 1)/len(tags_per_img)*100:.1f}%)")
    logger.info(f"   - Multi-tag images  (count>1):   {sum(1 for c in tags_per_img if c > 1)} ({sum(1 for c in tags_per_img if c > 1)/len(tags_per_img)*100:.1f}%)")
    logger.info(f"   - Max tags in a single image:    {max(tags_per_img) if tags_per_img else 0}")
    logger.info(f"   - Mean tags per image:           {np.mean(tags_per_img):.2f}")
    logger.info("=" * 70)


In [ ]:
# CELL 13 - Print Sample Images per Tag (In-Memory Display - No Files Saved)
from PIL import Image, ImageDraw
from IPython.display import display
from collections import defaultdict
from pathlib import Path
import os, json

# High-contrast color palette for distinct multi-class boxes
SAMPLE_COLORS = [
    (230, 25, 75),   # Red
    (60, 180, 75),   # Green
    (255, 225, 25),  # Yellow
    (67, 99, 216),   # Blue
    (245, 130, 49),  # Orange
    (145, 30, 180),  # Purple
    (66, 212, 244),  # Cyan
    (240, 50, 230),  # Magenta
    (191, 239, 69),  # Lime
    (250, 190, 212), # Pink
    (70, 153, 144),  # Teal
    (220, 190, 255)  # Lavender
]

def _safe_cat_key(val, default=0):
    """Normalize any category_id (int, string, list, dict) into a hashable identifier."""
    if isinstance(val, (list, tuple)):
        val = val[0] if len(val) > 0 else default
    elif isinstance(val, dict):
        val = val.get("id", val.get("category_id", val.get("name", default)))
    if val is None:
        return default
    try:
        return int(val)
    except (ValueError, TypeError):
        return str(val)

def print_sample_images_per_tag(annotation_source=None, image_dir=None, max_display_width: int = 700):
    """
    Selects sample images covering each discovered category tag, annotates
    bounding boxes with colored badges in memory, and prints/displays them
    inline directly in the notebook without saving any files to disk.
    
    Accepts:
    - annotation_source: Path or string to COCO JSON file, OR loaded dict/list
    - image_dir: Path or directory where images are cached
    """
    print("=" * 75)
    print("[CELL 13] Printing Sample Images with Category Tags (No Disk Saving)")
    print("=" * 75)

    if image_dir is None:
        image_dir = IMAGES_DIR if "IMAGES_DIR" in globals() else "images"
    image_dir = str(image_dir)

    categories = {}
    img_id_map = {}
    cat_to_images = defaultdict(list)
    img_to_anns = defaultdict(list)

    # Load annotations whether passed as path, dict, or list of records
    data = None
    if isinstance(annotation_source, (str, Path)) and os.path.exists(str(annotation_source)):
        with open(str(annotation_source), "r", encoding="utf-8") as f:
            data = json.load(f)
    elif isinstance(annotation_source, (dict, list)):
        data = annotation_source
    elif "image_annotations" in globals() and isinstance(globals()["image_annotations"], dict):
        # Fallback to parsed in-memory image annotations
        data = None

    if isinstance(data, dict):
        for c in data.get("categories", []):
            if isinstance(c, dict):
                cid = _safe_cat_key(c.get("id", 0))
                categories[cid] = str(c.get("name", cid))
        for im in data.get("images", []):
            if isinstance(im, dict):
                img_id_map[im.get("id")] = im
        for ann in data.get("annotations", []):
            if isinstance(ann, dict):
                cid = _safe_cat_key(ann.get("category_id", 0))
                iid = ann.get("image_id")
                cat_to_images[cid].append(iid)
                img_to_anns[iid].append(ann)
    elif isinstance(data, list):
        for idx, item in enumerate(data):
            if isinstance(item, dict):
                iid = item.get(globals().get("IMAGE_FIELD", "image_id"), idx)
                raw_cid = item.get("category_id", item.get(globals().get("CATEGORY_FIELD", "category_id"), 0))
                cid = _safe_cat_key(raw_cid)
                
                raw_cat_name = item.get("category_name", item.get(globals().get("CATEGORY_FIELD", "category_id"), f"class_{cid}"))
                if isinstance(raw_cat_name, (list, tuple)):
                    cat_name = str(raw_cat_name[0]) if len(raw_cat_name) > 0 else f"class_{cid}"
                elif isinstance(raw_cat_name, dict):
                    cat_name = str(raw_cat_name.get("name", raw_cat_name.get("id", f"class_{cid}")))
                else:
                    cat_name = str(raw_cat_name)
                    
                categories[cid] = cat_name
                fname = Path(str(item.get("image_url", item.get("coco_url", f"{iid}.jpg")))).name
                img_id_map[iid] = {"id": iid, "file_name": fname}
                cat_to_images[cid].append(iid)
                img_to_anns[iid].append(item)
    elif "image_annotations" in globals():
        in_mem = globals()["image_annotations"]
        in_cat_to_id = globals().get("cat_to_id", {})
        categories = {_safe_cat_key(idx): cat for cat, idx in in_cat_to_id.items()}
        for iid, anns in in_mem.items():
            fname = Path(str(iid)).name if "/" in str(iid) else f"{iid}.jpg"
            img_id_map[iid] = {"id": iid, "file_name": fname}
            for a in anns:
                cid = _safe_cat_key(a.get("category_id", 0))
                cat_to_images[cid].append(iid)
                img_to_anns[iid].append(a)

    if not categories and "AUTO_CATEGORIES" in globals():
        categories = {_safe_cat_key(idx): cat for idx, cat in enumerate(globals()["AUTO_CATEGORIES"])}

    # Match with existing local image files across all candidate directories
    search_dirs = [
        str(image_dir),
        str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
        str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
        str(Path("coco_files/images")),
        str(Path("images")),
        "/home/jupyter/multi_class_train_rfdetr/images",
        "/home/jupyter/coco_files/images",
        "/home/jupyter/images"
    ]
    search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

    existing_cat_images = {}
    for cid, iids in cat_to_images.items():
        valid_pairs = []
        for iid in iids:
            im_meta = img_id_map.get(iid, {})
            fname = im_meta.get("file_name", f"{iid}.jpg") if isinstance(im_meta, dict) else f"{iid}.jpg"
            
            candidate_names = [
                fname,
                Path(fname).name if fname else None,
                f"{iid}.jpg",
                f"{Path(str(iid)).name}",
                f"{Path(str(iid)).stem}.jpg"
            ]
            candidate_names = [n for n in candidate_names if n]

            found_path = None
            for d in search_dirs:
                for name in candidate_names:
                    p = os.path.join(d, name)
                    if os.path.exists(p) and os.path.getsize(p) > 0:
                        found_path = p
                        break
                if found_path:
                    break

            if found_path:
                valid_pairs.append((iid, found_path))
        if valid_pairs:
            existing_cat_images[cid] = valid_pairs

    if not existing_cat_images:
        print("[WARN] No matching local images found on disk to display yet.")
        print(f"       Searched image directories:")
        for d in search_dirs:
            count = len(os.listdir(d)) if os.path.exists(d) else 0
            print(f"         - {os.path.abspath(d)} ({count} files)")
        print("       (Run '00_download_coco_files.ipynb' first to download images into the shared pool)")
        return

    # Display an annotated sample for every distinct category tag (sort safely)
    sorted_cat_keys = sorted(existing_cat_images.keys(), key=lambda k: (0, k) if isinstance(k, int) else (1, str(k)))
    for cid in sorted_cat_keys:
        cat_name = categories.get(cid, str(cid))
        sample_img_id, img_path = existing_cat_images[cid][0]
        fname = os.path.basename(img_path)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[WARN] Could not open image {img_path}: {e}")
            continue

        im_w, im_h = img.size
        draw = ImageDraw.Draw(img)
        anns = img_to_anns.get(sample_img_id, [])

        cat_counts = defaultdict(int)
        for ann in anns:
            if not isinstance(ann, dict):
                continue
            raw_bbox = ann.get("bbox", [])
            if len(raw_bbox) != 4:
                continue
            try:
                x, y, w, h = [float(v) for v in raw_bbox]
            except (ValueError, TypeError):
                continue
            if w <= 0 or h <= 0:
                continue

            ann_cid = _safe_cat_key(ann.get("category_id", 0))
            ann_cname = str(categories.get(ann_cid, ann_cid))
            cat_counts[ann_cname] += 1
            color_idx = ann_cid if isinstance(ann_cid, int) else (hash(str(ann_cid)) & 0x7FFFFFFF)
            color = SAMPLE_COLORS[color_idx % len(SAMPLE_COLORS)]

            # Clamp box to valid dimensions
            x0 = max(0.0, min(float(im_w - 1), x))
            y0 = max(0.0, min(float(im_h - 1), y))
            x1 = max(0.0, min(float(im_w - 1), x + w))
            y1 = max(0.0, min(float(im_h - 1), y + h))

            # Draw bounding box
            draw.rectangle([x0, y0, x1, y1], outline=color, width=4)

            # Draw colored label badge with tag name
            label_text = f" {ann_cname} "
            if hasattr(draw, "textbbox"):
                tb = draw.textbbox((x0, max(0.0, y0 - 18.0)), label_text)
                draw.rectangle(tb, fill=color)
                draw.text((x0, max(0.0, y0 - 18.0)), label_text, fill=(255, 255, 255))
            else:
                draw.text((x0 + 2, max(0.0, y0 - 14.0)), label_text, fill=color)

        summary_str = ", ".join(f"'{k}': {v}" for k, v in sorted(cat_counts.items()))
        print(f"\n>>> Target Tag: [{cat_name}] | File: {fname} ({im_w}x{im_h}) | Boxes: {len(anns)} ({summary_str})")

        # Scale in-memory copy for clean notebook view if wide
        if im_w > max_display_width:
            scale = max_display_width / im_w
            display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
        else:
            display_img = img

        # Directly print/display in notebook (Zero files saved to disk)
        display(display_img)

    print("\n" + "=" * 75)
    print("All category tag samples displayed inline successfully.")
    print("=" * 75)

# Auto-discover first available annotation file or in-memory annotations
sample_ann_candidates = sorted(list(INPUT_JSON_DIR.glob("*.json"))) if "INPUT_JSON_DIR" in globals() else []
target_ann_file = sample_ann_candidates[0] if sample_ann_candidates else None

if target_ann_file and target_ann_file.exists():
    logger.info(f"Using '{target_ann_file.name}' for inline sample visualization...")
    print_sample_images_per_tag(str(target_ann_file), str(IMAGES_DIR))
else:
    # Use in-memory parsed annotations
    print_sample_images_per_tag(None, str(IMAGES_DIR))